In [12]:
import time
import requests
import pandas as pd

API_KEY = "RGAPI-768e19f0-bef8-4f41-a444-7480395a0299".strip()

SEA_PLATFORMS = {
    "SG2": "Singapore / Philippines / Thailand",
    "TW2": "Taiwan",
    "VN2": "Vietnam",
}

QUEUE = "RANKED_SOLO_5x5"
PLAYERS_PER_REGION = 20

In [16]:
def riot_get(url, max_retries=5):
    headers = {
        "X-Riot-Token": API_KEY,
        "Accept": "application/json",
    }

    for attempt in range(1, max_retries + 1):
        response = requests.get(
            url,
            headers=headers,
            timeout=30,
        )

        if response.status_code == 200:
            return response.json()

        if response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", 2))
            print(f"Rate limited. Waiting {retry_after} seconds...")
            time.sleep(retry_after)
            continue

        if response.status_code in {500, 502, 503, 504}:
            wait_time = attempt * 2
            print(
                f"Riot server error {response.status_code}. "
                f"Retrying in {wait_time} seconds..."
            )
            time.sleep(wait_time)
            continue

        raise RuntimeError(
            f"Request failed\n"
            f"Status: {response.status_code}\n"
            f"URL: {url}\n"
            f"Response: {response.text[:500]}"
        )

    raise RuntimeError("Maximum retries exceeded.")

In [17]:
test_url = (
    "https://sg2.api.riotgames.com"
    "/lol/league/v4/challengerleagues/by-queue/"
    "RANKED_SOLO_5x5"
)

test_data = riot_get(test_url)

test_data.keys()

dict_keys(['tier', 'queue', 'entries'])

In [18]:
sg2_entries = sorted(
    test_data["entries"],
    key=lambda player: player.get("leaguePoints", 0),
    reverse=True,
)[:20]

sg2_rows = []

for position, player in enumerate(sg2_entries, start=1):
    wins = player.get("wins", 0)
    losses = player.get("losses", 0)
    games_played = wins + losses

    sg2_rows.append({
        "platform": "SG2",
        "region": "Singapore / Philippines / Thailand",
        "position": position,
        "puuid": player.get("puuid"),
        "summoner_id": player.get("summonerId"),
        "rank": player.get("rank"),
        "league_points": player.get("leaguePoints", 0),
        "wins": wins,
        "losses": losses,
        "games_played": games_played,
        "win_rate": round(
            wins / games_played * 100,
            2,
        ) if games_played else 0,
        "hot_streak": player.get("hotStreak", False),
    })

df_sg2 = pd.DataFrame(sg2_rows)

df_sg2

,platform,region,position,puuid,summoner_id,rank,league_points,wins,losses,games_played,win_rate,hot_streak
0,SG2,Singapore / Philippines / Thailand,1,CYHXF3JglCMksGnfzOJHNP6rA_Cg-OwLJFUyY4s18TvzBb...,None,I,2706,276,155,431,64.04,False
1,SG2,Singapore / Philippines / Thailand,2,409Xdc_OYG4J-h3aJahviEJtmWqwzshfh_ct6okWzKk8vq...,None,I,2454,301,214,515,58.45,False
2,SG2,Singapore / Philippines / Thailand,3,BVIFcZB-UDTySogrrRQ_kvcedaQsoRcManQze9mAOqpj07...,None,I,2347,352,223,575,61.22,False
3,SG2,Singapore / Philippines / Thailand,4,4O1ayC5liHlnWhWENkFScM3DJLBtcpHK21Mk5jEl6aNJbI...,None,I,2229,363,303,666,54.50,False
4,SG2,Singapore / Philippines / Thailand,5,SMkjn6yMBOV3Z_o3SMeUhzpLBHdikGl5xSdoJn9-3rpAWM...,None,I,2227,311,203,514,60.51,False
5,SG2,Singapore / Philippines / Thailand,6,GPI98qiQNFstOjtbdEbO_fNyQlyauHeDxhKn_YENRaro_W...,None,I,2218,222,130,352,63.07,False
6,SG2,Singapore / Philippines / Thailand,7,KnwS2ySfEsEE7YQY9Rx1PFAW3TPz0vRAlLAAAHg2uSNUyb...,None,I,2187,280,201,481,58.21,False
7,SG2,Singapore / Philippines / Thailand,8,8rYP8gp8kuvsHqnfSHI-wuJEcJ_I1LoNhPefG8tazZ0wuC...,None,I,2039,147,74,221,66.52,False
8,SG2,Singapore / Philippines / Thailand,9,yLinrXmSSmuqHENHp7B5EudNw_c27SASN12cNvWrwA6LQF...,None,I,2005,310,254,564,54.96,False
9,SG2,Singapore / Philippines / Thailand,10,kkH8RQf5_yoLIt11fKlu5VX2yZIIZTJKMv7RvTFv6gYW27...,None,I,2005,269,207,476,56.51,False


In [21]:
def get_top_players(platform, region_name, limit=20):
    url = (
        f"https://{platform.lower()}.api.riotgames.com"
        f"/lol/league/v4/challengerleagues/by-queue/{QUEUE}"
    )

    leaderboard = riot_get(url)

    entries = sorted(
        leaderboard.get("entries", []),
        key=lambda player: player.get("leaguePoints", 0),
        reverse=True,
    )[:limit]

    rows = []

    for position, player in enumerate(entries, start=1):
        wins = player.get("wins", 0)
        losses = player.get("losses", 0)
        games_played = wins + losses

        rows.append({
            "platform": platform,
            "region": region_name,
            "position": position,
            "puuid": player.get("puuid"),
            "summoner_id": player.get("summonerId"),
            "tier": leaderboard.get("tier"),
            "rank": player.get("rank"),
            "league_points": player.get("leaguePoints", 0),
            "wins": wins,
            "losses": losses,
            "games_played": games_played,
            "win_rate": round(
                wins / games_played * 100,
                2,
            ) if games_played else 0,
            "hot_streak": player.get("hotStreak", False),
        })

    return rows

all_players = []

for platform, region_name in SEA_PLATFORMS.items():
    print(f"Calling {platform}...")

    region_players = get_top_players(
        platform=platform,
        region_name=region_name,
        limit=PLAYERS_PER_REGION,
    )

    all_players.extend(region_players)

df = pd.DataFrame(all_players)


pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)

df[
    [
        "platform",
        "position",
        "puuid",
        "league_points",
        "wins",
        "losses",
        "games_played",
        "win_rate",
    ]
]

Calling SG2...
Calling TW2...
Calling VN2...


,platform,position,puuid,league_points,wins,losses,games_played,win_rate
0,SG2,1,CYHXF3JglCMksGnfzOJHNP6rA_Cg-OwLJFUyY4s18TvzBb...,2706,276,155,431,64.04
1,SG2,2,409Xdc_OYG4J-h3aJahviEJtmWqwzshfh_ct6okWzKk8vq...,2454,301,214,515,58.45
2,SG2,3,BVIFcZB-UDTySogrrRQ_kvcedaQsoRcManQze9mAOqpj07...,2347,352,223,575,61.22
3,SG2,4,4O1ayC5liHlnWhWENkFScM3DJLBtcpHK21Mk5jEl6aNJbI...,2229,363,303,666,54.50
4,SG2,5,SMkjn6yMBOV3Z_o3SMeUhzpLBHdikGl5xSdoJn9-3rpAWM...,2227,311,203,514,60.51
5,SG2,6,GPI98qiQNFstOjtbdEbO_fNyQlyauHeDxhKn_YENRaro_W...,2218,222,130,352,63.07
6,SG2,7,KnwS2ySfEsEE7YQY9Rx1PFAW3TPz0vRAlLAAAHg2uSNUyb...,2187,280,201,481,58.21
7,SG2,8,8rYP8gp8kuvsHqnfSHI-wuJEcJ_I1LoNhPefG8tazZ0wuC...,2039,147,74,221,66.52
8,SG2,9,yLinrXmSSmuqHENHp7B5EudNw_c27SASN12cNvWrwA6LQF...,2005,310,254,564,54.96
9,SG2,10,kkH8RQf5_yoLIt11fKlu5VX2yZIIZTJKMv7RvTFv6gYW27...,2005,269,207,476,56.51


In [23]:
player = requests.get(
    "https://asia.api.riotgames.com/riot/account/v1/accounts/by-riot-id/burberry/5234",
    headers={"X-Riot-Token": API_KEY}
).json()

player

{'puuid': 'DLDP5bWITVDgoXEGAKjd6oQbgZ1_wWSGbQQkK2YLISUS4LR7iCxWHAeIuyBBObMuuGbnoELCou_oxw',
 'gameName': 'burberry',
 'tagLine': '5234'}